# Document Extraction Inspection

## Purpose

This notebook inspects the Version 1 source documents before building the ingestion pipeline.

We will first confirm the expected files and their sizes. Later sections will compare PDF extraction methods, page coverage, reading order, repeated headers, bilingual duplication, tables, and charts.

The raw PDFs must remain unchanged.

In [ ]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

if not raw_data_dir.exists():
    raise FileNotFoundError(f"Raw-data directory not found: {raw_data_dir}")

pdf_paths = sorted(raw_data_dir.glob("*.pdf"))

document_inventory = pd.DataFrame(
    {
        "file_name": [path.name for path in pdf_paths],
        "size_mb": [round(path.stat().st_size / (1024**2), 2) for path in pdf_paths],
    }
)

print(f"Project root: {project_root}")
print(f"PDF count: {len(pdf_paths)}")

document_inventory

## Baseline Text-Layer Inspection

We first test PyMuPDF because it can extract text page by page while preserving page boundaries.

This check measures whether each PDF already contains searchable text. It does not yet prove that reading order, tables, bilingual content, or citations are correct.

In [ ]:
import fitz

extraction_records = []

for pdf_path in pdf_paths:
    with fitz.open(pdf_path) as document:
        page_character_counts = [
            len(page.get_text("text").strip())
            for page in document
        ]

    text_page_count = sum(count > 0 for count in page_character_counts)

    extraction_records.append(
        {
            "file_name": pdf_path.name,
            "page_count": len(page_character_counts),
            "text_pages": text_page_count,
            "empty_text_pages": len(page_character_counts) - text_page_count,
            "total_characters": sum(page_character_counts),
            "median_characters_per_page": round(
                pd.Series(page_character_counts).median()
            ),
        }
    )

text_layer_summary = pd.DataFrame(extraction_records)

print(text_layer_summary)

## Extraction Method Comparison

All documents contain searchable text, so OCR is unnecessary.

We now compare PyMuPDF, pypdf, and pdfplumber on a report page containing narrative text and tabular information. The goal is to inspect reading order, missing text, duplicated text, and whether labels remain connected to their values.

In [ ]:
from pypdf import PdfReader
import pdfplumber

# Compare mana yang lagi better based on libraries yang kita decide. 
report_path = (
    raw_data_dir
    / "2H24-Consumer_Report_Jul-to-Dec-2024-Final_CCR.pdf"
)

page_number = 5
page_index = page_number - 1

with fitz.open(report_path) as document:
    pymupdf_text = document[page_index].get_text("text")

pypdf_reader = PdfReader(report_path)
pypdf_text = pypdf_reader.pages[page_index].extract_text() or ""

with pdfplumber.open(report_path) as document:
    pdfplumber_text = document.pages[page_index].extract_text() or ""

extraction_comparison = {
    "PyMuPDF": pymupdf_text,
    "pypdf": pypdf_text,
    "pdfplumber": pdfplumber_text,
}

for method, extracted_text in extraction_comparison.items():
    print(f"\n{'=' * 20} {method} {'=' * 20}\n")
    print(extracted_text[:2000])

## Block-Level Layout Inspection

Plain text extraction preserves narrative paragraphs but destroys relationships inside the chart.

We now inspect positioned text blocks from PyMuPDF. Coordinates show where each text block appears on the page. This tests whether basic layout information can separate narrative text, chart content, sources, and repeated footers without training a layout model.

In [ ]:
with fitz.open(report_path) as document:
    page = document[page_index]
    page_width = page.rect.width
    page_height = page.rect.height
    blocks = page.get_text("blocks")

block_records = []

for x0, y0, x1, y1, text, block_number, block_type in blocks:
    cleaned_text = " | ".join(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )

    if not cleaned_text:
        continue

    block_records.append(
        {
            "block": block_number,
            "x0": round(x0, 1),
            "y0": round(y0, 1),
            "x1": round(x1, 1),
            "y1": round(y1, 1),
            "text": cleaned_text,
        }
    )

block_summary = (
    pd.DataFrame(block_records)
    .sort_values(["y0", "x0"])
    .reset_index(drop=True)
)

print(f"Page size: {page_width:.1f} x {page_height:.1f}")
print(block_summary.to_string(index=False))

## Interpretation

All three libraries extracted the narrative paragraphs successfully, but none reconstructed the chart as structured rows and columns.

This happens because the page contains a visual bar chart rather than a real PDF table. The PDF stores text labels, numbers, shapes, and coordinates separately. It does not store the semantic relationship between an airline, reporting period, passenger volume, and complaint count.

PyMuPDF blocks preserve coordinates, but coordinates alone do not reveal the meaning of each chart element.

Decision:

- Use PyMuPDF as the baseline extractor for narrative text.
- Preserve page numbers and block coordinates.
- Remove repeated headers and footers during cleaning.
- Do not use raw chart text for numerical answers.
- Later, store selected chart values in a manually validated structured dataset if those values are required.

## Bilingual Gazette Inspection

The Gazette documents contain Malay and English versions of the same legal content.

Indexing both versions without a clear strategy could create near-duplicate chunks and weaken retrieval. We first identify where the English sections begin.

In [ ]:
legal_documents = {
    "principal_2016": raw_data_dir / "pub_20160630_P.U.-B-305.pdf",
    "amendment_2019": (
        raw_data_dir
        / "Malaysian-Aviation-Consumer-Protection-Amendment-Code-2019.pdf"
    ),
    "amendment_2024": (
        raw_data_dir
        / "MACPC-Enhancement-PUB345_2024.pdf"
    ),
}

english_marker = "MALAYSIAN AVIATION COMMISSION ACT 2015"

for document_id, pdf_path in legal_documents.items():
    matching_pages = []

    with fitz.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            page_text = page.get_text("text").upper()

            if english_marker in page_text:
                matching_pages.append(page_number)

    print(f"{document_id}: {matching_pages}")

## Inspecting Language Boundaries

The English marker identifies legal-body pages, not necessarily the first page of the English section.

We inspect several pages around the expected language boundary before deciding which pages belong to each language.

In [ ]:
boundary_pages = {
    "principal_2016": range(33, 40),
    "amendment_2019": range(14, 20),
    "amendment_2024": range(29, 35),
}

for document_id, page_numbers in boundary_pages.items():
    print(f"\n{'=' * 15} {document_id} {'=' * 15}")

    pdf_path = legal_documents[document_id]

    with fitz.open(pdf_path) as document:
        for page_number in page_numbers:
            page_text = document[page_number - 1].get_text("text")

            first_lines = [
                line.strip()
                for line in page_text.splitlines()
                if line.strip()
            ][:6]

            preview = " | ".join(first_lines)

            print(f"PDF page {page_number}: {preview}")

## English Scope And Repeated-Line Inspection

Version 1 will index only the English Gazette sections to avoid bilingual duplicate chunks.

We now count repeated lines near the top and bottom of pages. Frequent lines may be page headers or footers that should be removed before chunking. Frequency alone does not justify deletion, so candidates must be reviewed.

In [ ]:
from collections import Counter


english_page_ranges = {
    "principal_2016": (37, 66),
    "amendment_2019": (19, 32),
    "amendment_2024": (34, 58),
}

for document_id, (start_page, end_page) in english_page_ranges.items():
    pdf_path = legal_documents[document_id]

    top_line_counts = Counter()
    bottom_line_counts = Counter()

    with fitz.open(pdf_path) as document:
        for page_number in range(start_page, end_page + 1):
            page_text = document[page_number - 1].get_text("text")

            lines = [
                line.strip()
                for line in page_text.splitlines()
                if line.strip()
            ]

            top_line_counts.update(lines[:3])
            bottom_line_counts.update(lines[-3:])

    print(f"\n{'=' * 15} {document_id} {'=' * 15}")
    print(f"English page range: {start_page}-{end_page}")

    print("\nMost common top lines:")
    for line, count in top_line_counts.most_common(8):
        print(f"{count:>3} | {line}")

    print("\nMost common bottom lines:")
    for line, count in bottom_line_counts.most_common(8):
        print(f"{count:>3} | {line}")

## Gazette Header Cleaning

The Gazette identifier and page number repeat at the top of every legal page. They add noise but no useful meaning for retrieval.

We remove them only when they appear at the beginning of a page. We do not remove repeated legal labels such as `(a)` or `(b)`.

In [ ]:
import re


gazette_header_pattern = re.compile(
    r"^P\.U\. \(B\) \d+$"
)


def clean_gazette_page_text(text, pdf_page_number):
    """Remove a Gazette header and its page number from one page."""

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    removed_lines = []

    if lines and gazette_header_pattern.fullmatch(lines[0]):
        removed_lines.append(lines.pop(0))

    if lines and lines[0] == str(pdf_page_number):
        removed_lines.append(lines.pop(0))

    cleaned_text = "\n".join(lines)

    return cleaned_text, removed_lines

In [ ]:
sample_pages = {
    "principal_2016": 39,
    "amendment_2019": 19,
    "amendment_2024": 34,
}

for document_id, page_number in sample_pages.items():
    pdf_path = legal_documents[document_id]

    with fitz.open(pdf_path) as document:
        raw_text = document[page_number - 1].get_text("text")

    cleaned_text, removed_lines = clean_gazette_page_text(
        raw_text,
        page_number,
    )

    print(f"\n{'=' * 15} {document_id} {'=' * 15}")
    print(f"Removed: {removed_lines}")
    print("\nCleaned beginning:")
    print("\n".join(cleaned_text.splitlines()[:8]))

## Legal-Text Extraction Comparison

The Gazette header was removed successfully, but PyMuPDF split part of the legal sentence into individual lines.

We compare the three extraction libraries on the same legal page before deciding whether cleaning logic is needed.

In [ ]:
legal_path = legal_documents["principal_2016"]
legal_page_number = 39
legal_page_index = legal_page_number - 1

with fitz.open(legal_path) as document:
    pymupdf_legal_text = document[legal_page_index].get_text("text")

pypdf_reader = PdfReader(legal_path)
pypdf_legal_text = (
    pypdf_reader.pages[legal_page_index].extract_text()
    or ""
)

with pdfplumber.open(legal_path) as document:
    pdfplumber_legal_text = (
        document.pages[legal_page_index].extract_text()
        or ""
    )

legal_extraction_comparison = {
    "PyMuPDF": pymupdf_legal_text,
    "pypdf": pypdf_legal_text,
    "pdfplumber": pdfplumber_legal_text,
}

for method, extracted_text in legal_extraction_comparison.items():
    cleaned_text, removed_lines = clean_gazette_page_text(
        extracted_text,
        legal_page_number,
    )

    print(f"\n{'=' * 20} {method} {'=' * 20}")
    print(f"Removed: {removed_lines}")
    print(cleaned_text[:1500])

## Extractor Selection

No extraction library performed best on every document type.

For Gazette documents, pdfplumber preserved legal sentences and words better than PyMuPDF and pypdf. For consumer reports, PyMuPDF produced cleaner narrative reading order.

The ingestion pipeline will route documents to an extractor based on document type.

In [ ]:
def clean_gazette_page_text(text, pdf_page_number):
    """Remove Gazette identifiers and page numbers from one page."""

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    removed_lines = []

    if lines and gazette_header_pattern.fullmatch(lines[0]):
        removed_lines.append(lines.pop(0))

    page_number_text = str(pdf_page_number)

    if lines and lines[0] == page_number_text:
        removed_lines.append(lines.pop(0))

    if lines and lines[-1] == page_number_text:
        removed_lines.append(lines.pop())

    cleaned_text = "\n".join(lines)

    return cleaned_text, removed_lines

In [ ]:
sample_pages = {
    "principal_2016": 39,
    "amendment_2019": 19,
    "amendment_2024": 34,
}

for document_id, page_number in sample_pages.items():
    pdf_path = legal_documents[document_id]

    with pdfplumber.open(pdf_path) as document:
        raw_text = (
            document.pages[page_number - 1].extract_text()
            or ""
        )

    cleaned_text, removed_lines = clean_gazette_page_text(
        raw_text,
        page_number,
    )

    print(f"\n{'=' * 15} {document_id} {'=' * 15}")
    print(f"Removed: {removed_lines}")
    print(cleaned_text[:500])

## Consumer Report Repeated-Line Inspection

Consumer reports use different templates across reporting periods.

We count repeated lines near page boundaries to identify disclaimers, report titles, and page markers that may pollute retrieval.

In [ ]:
report_documents = {
    "2H2023": (
        raw_data_dir
        / "Consumer-Report-July-2023-to-December-2023-FINAL.pdf"
    ),
    "1H2024": (
        raw_data_dir
        / "Appendix-I_-Consumer-Report-Jan-to-Jun-2024.pdf"
    ),
    "2H2024": (
        raw_data_dir
        / "2H24-Consumer_Report_Jul-to-Dec-2024-Final_CCR.pdf"
    ),
    "1H2025": (
        raw_data_dir
        / "Consumer-Report-Jan-Jun-2025.pdf"
    ),
}

for document_id, pdf_path in report_documents.items():
    top_line_counts = Counter()
    bottom_line_counts = Counter()

    with fitz.open(pdf_path) as document:
        for page in document:
            lines = [
                line.strip()
                for line in page.get_text("text").splitlines()
                if line.strip()
            ]

            top_line_counts.update(lines[:3])
            bottom_line_counts.update(lines[-3:])

    print(f"\n{'=' * 15} {document_id} {'=' * 15}")

    print("\nMost common top lines:")
    for line, count in top_line_counts.most_common(6):
        print(f"{count:>3} | {line}")

    print("\nMost common bottom lines:")
    for line, count in bottom_line_counts.most_common(6):
        print(f"{count:>3} | {line}")

## Consumer Report Cleaning Rules

Repeated distribution disclaimers and report headers add retrieval noise.

We remove only exact verified header strings. Repeated years, percentages, and numbers remain because they may belong to charts or tables.

In [ ]:
report_disclaimer_lines = {
    (
        "DO NOT DUPLICATE OR DISTRIBUTE WITHOUT WRITTEN "
        "PERMISSION FROM MALAYSIAN AVIATION COMMISSION"
    ),
    (
        "PRIVATE & CONFIDENTIAL - DO NOT DUPLICATE OR "
        "DISTRIBUTE WITHOUT WRITTEN PERMISSION FROM "
        "MALAYSIAN AVIATION COMMISSION"
    ),
}


def clean_report_page_text(
    text,
    document_id,
    pdf_page_number,
):
    """Remove verified repeated noise from a report page."""

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    cleaned_lines = []
    removed_lines = []

    for line_index, line in enumerate(lines):
        if line in report_disclaimer_lines:
            removed_lines.append(line)
            continue

        if (
            line_index < 3
            and line == "CONSUMER REPORT"
        ):
            removed_lines.append(line)
            continue

        cleaned_lines.append(line)

    page_number_text = str(pdf_page_number)

    if cleaned_lines and cleaned_lines[0] == page_number_text:
        removed_lines.append(cleaned_lines.pop(0))

    if cleaned_lines and cleaned_lines[-1] == page_number_text:
        removed_lines.append(cleaned_lines.pop())

    return "\n".join(cleaned_lines), removed_lines

In [ ]:
quality_records = []

for document_id, pdf_path in report_documents.items():
    with fitz.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            raw_text = page.get_text("text")

            cleaned_text, removed_lines = clean_report_page_text(
                raw_text,
                document_id,
                page_number,
            )

            cleaned_lines = cleaned_text.splitlines()
            page_number_text = str(page_number)

            quality_records.append(
                {
                    "document_id": document_id,
                    "page_number": page_number,
                    "empty_after_cleaning": not cleaned_text.strip(),
                    "page_number_at_start": (
                        bool(cleaned_lines)
                        and cleaned_lines[0] == page_number_text
                    ),
                    "page_number_at_end": (
                        bool(cleaned_lines)
                        and cleaned_lines[-1] == page_number_text
                    ),
                    "remaining_disclaimer": any(
                        disclaimer in cleaned_text
                        for disclaimer in report_disclaimer_lines
                    ),
                }
            )

quality_results = pd.DataFrame(quality_records)

quality_summary = quality_results.groupby("document_id").agg(
    pages=("page_number", "count"),
    empty_pages=("empty_after_cleaning", "sum"),
    page_number_at_start=("page_number_at_start", "sum"),
    page_number_at_end=("page_number_at_end", "sum"),
    remaining_disclaimer=("remaining_disclaimer", "sum"),
)

print(quality_summary)

In [ ]:
legal_quality_records = []

for document_id, (start_page, end_page) in english_page_ranges.items():
    pdf_path = legal_documents[document_id]

    with pdfplumber.open(pdf_path) as document:
        for page_number in range(start_page, end_page + 1):
            raw_text = (
                document.pages[page_number - 1].extract_text()
                or ""
            )

            cleaned_text, removed_lines = clean_gazette_page_text(
                raw_text,
                page_number,
            )

            cleaned_lines = cleaned_text.splitlines()
            page_number_text = str(page_number)

            legal_quality_records.append(
                {
                    "document_id": document_id,
                    "page_number": page_number,
                    "empty_after_cleaning": not cleaned_text.strip(),
                    "header_at_start": (
                        bool(cleaned_lines)
                        and gazette_header_pattern.fullmatch(
                            cleaned_lines[0]
                        )
                        is not None
                    ),
                    "page_number_at_start": (
                        bool(cleaned_lines)
                        and cleaned_lines[0] == page_number_text
                    ),
                    "page_number_at_end": (
                        bool(cleaned_lines)
                        and cleaned_lines[-1] == page_number_text
                    ),
                }
            )

legal_quality_results = pd.DataFrame(
    legal_quality_records
)

legal_quality_summary = (
    legal_quality_results
    .groupby("document_id")
    .agg(
        indexed_pages=("page_number", "count"),
        empty_pages=("empty_after_cleaning", "sum"),
        header_at_start=("header_at_start", "sum"),
        page_number_at_start=("page_number_at_start", "sum"),
        page_number_at_end=("page_number_at_end", "sum"),
    )
)

print(legal_quality_summary)

## Technical Document Manifest

A manifest records every source file used by the project.

The SHA-256 hash acts like a file fingerprint. If a source PDF changes later, its hash changes, allowing us to detect that the evaluation may no longer use the same document version.

In [ ]:
import hashlib
from datetime import date


def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """Calculate a SHA-256 fingerprint without loading the full file."""

    hasher = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            hasher.update(chunk)

    return hasher.hexdigest()


manifest_records = []

for pdf_path in pdf_paths:
    with fitz.open(pdf_path) as document:
        page_count = document.page_count

    is_guidance = (
        pdf_path.name
        == "MACPC-2016-ENG-Updated-24-Apr-2025.pdf"
    )

    manifest_records.append(
        {
            "file_name": pdf_path.name,
            "file_size_bytes": pdf_path.stat().st_size,
            "page_count": page_count,
            "sha256": calculate_sha256(pdf_path),
            "access_date": date.today().isoformat(),
            "include_in_v1": not is_guidance,
        }
    )

technical_manifest = pd.DataFrame(manifest_records)

manifest_path = (
    project_root
    / "data"
    / "manifests"
    / "technical_manifest.csv"
)

technical_manifest.to_csv(
    manifest_path,
    index=False,
)

print(f"Manifest saved to: {manifest_path}")
print(technical_manifest.to_string(index=False))

## Semantic Document Metadata

Technical metadata identifies a file. Semantic metadata explains its role, authority, time coverage, language, and extraction method.

Retrieval will later use these fields for filtering and citations.

In [ ]:
semantic_records = [
    {
        "document_id": "report_2h2024",
        "file_name": "2H24-Consumer_Report_Jul-to-Dec-2024-Final_CCR.pdf",
        "document_type": "consumer_report",
        "title": "Consumer Report July to December 2024",
        "issuer": "MAVCOM",
        "authority_level": "official_report",
        "reporting_period_start": "2024-07-01",
        "reporting_period_end": "2024-12-31",
        "effective_from": None,
        "indexed_language": "en",
        "extraction_method": "pymupdf",
        "indexed_start_page": 1,
        "indexed_end_page": 35,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "2H24-Consumer_Report_Jul-to-Dec-2024-Final_CCR.pdf"
        ),
        "redistribution_status": "restricted_notice_in_pdf",
    },
    {
        "document_id": "report_1h2024",
        "file_name": "Appendix-I_-Consumer-Report-Jan-to-Jun-2024.pdf",
        "document_type": "consumer_report",
        "title": "Consumer Report January to June 2024",
        "issuer": "MAVCOM",
        "authority_level": "official_report",
        "reporting_period_start": "2024-01-01",
        "reporting_period_end": "2024-06-30",
        "effective_from": None,
        "indexed_language": "en",
        "extraction_method": "pymupdf",
        "indexed_start_page": 1,
        "indexed_end_page": 22,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "Appendix-I_-Consumer-Report-Jan-to-Jun-2024.pdf"
        ),
        "redistribution_status": "restricted_notice_in_pdf",
    },
    {
        "document_id": "report_1h2025",
        "file_name": "Consumer-Report-Jan-Jun-2025.pdf",
        "document_type": "consumer_report",
        "title": "Consumer Report January to June 2025",
        "issuer": "CAAM",
        "authority_level": "official_report",
        "reporting_period_start": "2025-01-01",
        "reporting_period_end": "2025-06-30",
        "effective_from": None,
        "indexed_language": "en",
        "extraction_method": "pymupdf",
        "indexed_start_page": 1,
        "indexed_end_page": 53,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/12/"
            "Consumer-Report-Jan-Jun-2025.pdf"
        ),
        "redistribution_status": "requires_external_terms_review",
    },
    {
        "document_id": "report_2h2023",
        "file_name": (
            "Consumer-Report-July-2023-to-December-2023-FINAL.pdf"
        ),
        "document_type": "consumer_report",
        "title": "Consumer Report July to December 2023",
        "issuer": "MAVCOM",
        "authority_level": "official_report",
        "reporting_period_start": "2023-07-01",
        "reporting_period_end": "2023-12-31",
        "effective_from": None,
        "indexed_language": "en",
        "extraction_method": "pymupdf",
        "indexed_start_page": 1,
        "indexed_end_page": 22,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "Consumer-Report-July-2023-to-December-2023-FINAL.pdf"
        ),
        "redistribution_status": "restricted_notice_in_pdf",
    },
    {
        "document_id": "guidance_2025",
        "file_name": "MACPC-2016-ENG-Updated-24-Apr-2025.pdf",
        "document_type": "consumer_guidance",
        "title": (
            "Malaysian Aviation Consumer Protection Code 2016: "
            "Know Your Rights"
        ),
        "issuer": "MAVCOM",
        "authority_level": "official_guidance",
        "reporting_period_start": None,
        "reporting_period_end": None,
        "effective_from": None,
        "indexed_language": "en",
        "extraction_method": "pymupdf",
        "indexed_start_page": None,
        "indexed_end_page": None,
        "source_url": (
            "https://flysmart.my/wp-content/uploads/2025/04/"
            "MACPC-2016-ENG-Updated-24-Apr-2025.pdf"
        ),
        "redistribution_status": "requires_external_terms_review",
    },
    {
        "document_id": "macpc_amendment_2024",
        "file_name": "MACPC-Enhancement-PUB345_2024.pdf",
        "document_type": "gazette_amendment",
        "title": (
            "Malaysian Aviation Consumer Protection "
            "(Amendment) Code 2024"
        ),
        "issuer": "MAVCOM",
        "authority_level": "gazette",
        "reporting_period_start": None,
        "reporting_period_end": None,
        "effective_from": "2024-09-01",
        "indexed_language": "en",
        "extraction_method": "pdfplumber",
        "indexed_start_page": 34,
        "indexed_end_page": 58,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "MACPC-Enhancement-PUB345_2024.pdf"
        ),
        "redistribution_status": "requires_external_terms_review",
    },
    {
        "document_id": "macpc_amendment_2019",
        "file_name": (
            "Malaysian-Aviation-Consumer-Protection-"
            "Amendment-Code-2019.pdf"
        ),
        "document_type": "gazette_amendment",
        "title": (
            "Malaysian Aviation Consumer Protection "
            "(Amendment) Code 2019"
        ),
        "issuer": "MAVCOM",
        "authority_level": "gazette",
        "reporting_period_start": None,
        "reporting_period_end": None,
        "effective_from": "2019-06-01",
        "indexed_language": "en",
        "extraction_method": "pdfplumber",
        "indexed_start_page": 19,
        "indexed_end_page": 32,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "Malaysian-Aviation-Consumer-Protection-"
            "Amendment-Code-2019.pdf"
        ),
        "redistribution_status": "requires_external_terms_review",
    },
    {
        "document_id": "macpc_principal_2016",
        "file_name": "pub_20160630_P.U.-B-305.pdf",
        "document_type": "gazette_principal",
        "title": "Malaysian Aviation Consumer Protection Code 2016",
        "issuer": "MAVCOM",
        "authority_level": "gazette",
        "reporting_period_start": None,
        "reporting_period_end": None,
        "effective_from": "2016-07-01",
        "indexed_language": "en",
        "extraction_method": "pdfplumber",
        "indexed_start_page": 37,
        "indexed_end_page": 66,
        "source_url": (
            "https://www.caam.gov.my/wp-content/uploads/2025/07/"
            "pub_20160630_P.U.-B-305.pdf"
        ),
        "redistribution_status": "requires_external_terms_review",
    },
]

semantic_metadata = pd.DataFrame(semantic_records)

document_manifest = technical_manifest.merge(
    semantic_metadata,
    on="file_name",
    how="left",
    validate="one_to_one",
)

assert len(document_manifest) == 8
assert document_manifest["document_id"].notna().all()
assert document_manifest["document_id"].is_unique
assert document_manifest["source_url"].notna().all()
assert document_manifest["include_in_v1"].sum() == 7

document_manifest_path = (
    project_root
    / "data"
    / "manifests"
    / "documents.csv"
)

document_manifest.to_csv(
    document_manifest_path,
    index=False,
)

print(f"Document manifest saved to: {document_manifest_path}")

print(
    document_manifest[
        [
            "document_id",
            "document_type",
            "authority_level",
            "extraction_method",
            "include_in_v1",
        ]
    ].to_string(index=False)
)

In [ ]:
gazette_number_by_id = {
    "macpc_principal_2016": "P.U. (B) 305",
    "macpc_amendment_2019": "P.U. (B) 250",
    "macpc_amendment_2024": "P.U. (B) 345",
}

publication_date_by_id = {
    "macpc_principal_2016": "2016-06-30",
    "macpc_amendment_2019": "2019-05-10",
    "macpc_amendment_2024": "2024-08-30",
}

effective_date_note_by_id = {
    "macpc_principal_2016": (
        "Principal Code effective 2016-07-01."
    ),
    "macpc_amendment_2019": (
        "All provisions effective 2019-06-01."
    ),
    "macpc_amendment_2024": (
        "Most provisions effective 2024-09-01; "
        "new paragraph 8A effective 2025-01-01."
    ),
}

document_manifest["gazette_number"] = (
    document_manifest["document_id"]
    .map(gazette_number_by_id)
)

document_manifest["publication_date"] = (
    document_manifest["document_id"]
    .map(publication_date_by_id)
)

document_manifest["effective_date_note"] = (
    document_manifest["document_id"]
    .map(effective_date_note_by_id)
)

document_manifest.to_csv(
    document_manifest_path,
    index=False,
)

gazette_manifest = document_manifest[
    document_manifest["authority_level"] == "gazette"
][
    [
        "document_id",
        "gazette_number",
        "publication_date",
        "effective_from",
        "effective_date_note",
    ]
]

print(gazette_manifest.to_string(index=False))

## Page-Level Extraction Pipeline

Each included document is extracted page by page using the method selected during inspection.

Every page record keeps its document identity, authority, dates, source URL, page number, cleaned text, and removed noise. Page boundaries must survive because citations depend on them.

In [ ]:
def clean_nullable_value(value):
    """Convert pandas missing values into normal Python None."""

    return None if pd.isna(value) else value


def extract_document_pages(manifest_row):
    """Extract and clean all indexed pages for one document."""

    pdf_path = raw_data_dir / manifest_row["file_name"]

    start_page = int(manifest_row["indexed_start_page"])
    end_page = int(manifest_row["indexed_end_page"])

    extracted_pages = []

    if manifest_row["extraction_method"] == "pdfplumber":
        with pdfplumber.open(pdf_path) as document:
            for page_number in range(start_page, end_page + 1):
                raw_text = (
                    document.pages[page_number - 1].extract_text()
                    or ""
                )

                cleaned_text, removed_lines = (
                    clean_gazette_page_text(
                        raw_text,
                        page_number,
                    )
                )

                extracted_pages.append(
                    {
                        "page_number": page_number,
                        "text": cleaned_text,
                        "removed_lines": removed_lines,
                    }
                )

    elif manifest_row["extraction_method"] == "pymupdf":
        with fitz.open(pdf_path) as document:
            for page_number in range(start_page, end_page + 1):
                raw_text = document[
                    page_number - 1
                ].get_text("text")

                cleaned_text, removed_lines = (
                    clean_report_page_text(
                        raw_text,
                        manifest_row["document_id"],
                        page_number,
                    )
                )

                extracted_pages.append(
                    {
                        "page_number": page_number,
                        "text": cleaned_text,
                        "removed_lines": removed_lines,
                    }
                )

    else:
        raise ValueError(
            "Unsupported extraction method: "
            f"{manifest_row['extraction_method']}"
        )

    return extracted_pages

In [ ]:
page_records = []

included_manifest = document_manifest[
    document_manifest["include_in_v1"]
].copy()

for _, manifest_row in included_manifest.iterrows():
    extracted_pages = extract_document_pages(
        manifest_row
    )

    for page in extracted_pages:
        page_records.append(
            {
                "document_id": manifest_row["document_id"],
                "title": manifest_row["title"],
                "document_type": manifest_row["document_type"],
                "authority_level": manifest_row["authority_level"],
                "gazette_number": clean_nullable_value(
                    manifest_row["gazette_number"]
                ),
                "publication_date": clean_nullable_value(
                    manifest_row["publication_date"]
                ),
                "effective_from": clean_nullable_value(
                    manifest_row["effective_from"]
                ),
                "effective_date_note": clean_nullable_value(
                    manifest_row["effective_date_note"]
                ),
                "reporting_period_start": clean_nullable_value(
                    manifest_row["reporting_period_start"]
                ),
                "reporting_period_end": clean_nullable_value(
                    manifest_row["reporting_period_end"]
                ),
                "source_url": manifest_row["source_url"],
                "page_number": page["page_number"],
                "text": page["text"],
                "character_count": len(page["text"]),
                "removed_lines": page["removed_lines"],
            }
        )

page_records_df = pd.DataFrame(page_records)

expected_page_count = (
    included_manifest["indexed_end_page"]
    - included_manifest["indexed_start_page"]
    + 1
).sum()

assert len(page_records_df) == expected_page_count
assert not page_records_df["text"].str.strip().eq("").any()
assert not page_records_df.duplicated(
    ["document_id", "page_number"]
).any()

pages_output_path = (
    project_root
    / "data"
    / "processed"
    / "pages.jsonl"
)

page_records_df.to_json(
    pages_output_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"Page records saved to: {pages_output_path}")
print(f"Expected pages: {int(expected_page_count)}")
print(f"Extracted pages: {len(page_records_df)}")

print(
    page_records_df.groupby("document_id")
    .agg(
        pages=("page_number", "count"),
        total_characters=("character_count", "sum"),
        median_characters=("character_count", "median"),
    )
)

## Page-Length Outlier Inspection

Character counts help identify pages requiring manual review.

Very short pages may contain covers, charts, or extraction failures. Very long pages may contain dense tables, duplicated text, or several sections that should not become one retrieval unit.

In [ ]:
page_length_summary = (
    page_records_df
    .groupby("document_id")
    .agg(
        minimum_characters=("character_count", "min"),
        median_characters=("character_count", "median"),
        maximum_characters=("character_count", "max"),
    )
)

print("Page-length summary:")
print(page_length_summary)

preview_columns = [
    "document_id",
    "page_number",
    "character_count",
    "text_preview",
]

page_previews = page_records_df.copy()

page_previews["text_preview"] = (
    page_previews["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 180)
)

shortest_pages = (
    page_previews
    .sort_values("character_count")
    [preview_columns]
    .head(12)
)

longest_pages = (
    page_previews
    .sort_values("character_count", ascending=False)
    [preview_columns]
    .head(12)
)

print("\nShortest pages:")
print(shortest_pages.to_string(index=False))

print("\nLongest pages:")
print(longest_pages.to_string(index=False))

In [ ]:
#Saja Check
page_6_text = page_records_df.loc[
    (page_records_df["document_id"] == "report_1h2025")
    & (page_records_df["page_number"] == 6),
    "text",
].iloc[0]

print("\n".join(page_6_text.splitlines()[:5]))

assert not page_6_text.startswith("CONSUMER REPORT")
assert not page_6_text.startswith("6")

In [ ]:
short_page_candidates = page_records_df.loc[
    page_records_df["character_count"] < 100,
    [
        "document_id",
        "page_number",
        "character_count",
        "text",
    ],
].copy()

short_page_candidates["text"] = (
    short_page_candidates["text"]
    .str.replace(r"\s+", " ", regex=True)
)

print(f"Short-page count: {len(short_page_candidates)}")
print(short_page_candidates.to_string(index=False))

In [ ]:
#Saja Check Figure
from IPython.display import Image, Markdown, display


report_1h2025_path = report_documents["1H2025"]

with fitz.open(report_1h2025_path) as document:
    for page_number in [30, 32]:
        page = document[page_number - 1]

        pixmap = page.get_pixmap(
            matrix=fitz.Matrix(1.2, 1.2),
            alpha=False,
        )

        display(
            Markdown(f"### PDF Page {page_number}"),
            Image(data=pixmap.tobytes("png")),
        )

## Retrieval Page Eligibility

Short pages were visually reviewed.

Covers, section dividers, appendix dividers, thank-you pages, and image-only tables will remain in the extracted dataset but will not enter text retrieval. Keeping exclusion reasons makes the decision auditable.

In [ ]:
page_exclusions = {
    ("report_2h2024", 1): "cover",
    ("report_2h2024", 14): "section_divider",
    ("report_2h2024", 19): "section_divider",
    ("report_2h2024", 33): "appendix_divider",
    ("report_2h2024", 35): "thank_you_page",

    ("report_1h2024", 1): "cover",
    ("report_1h2024", 13): "section_divider",
    ("report_1h2024", 20): "appendix_divider",
    ("report_1h2024", 22): "thank_you_page",

    ("report_1h2025", 1): "cover",
    ("report_1h2025", 30): "image_table_requires_validation",
    ("report_1h2025", 32): "image_table_requires_validation",

    ("report_2h2023", 1): "cover",
    ("report_2h2023", 14): "section_divider",
    ("report_2h2023", 22): "thank_you_page",
}

page_records_df["retrieval_eligible"] = True
page_records_df["exclusion_reason"] = None

for (document_id, page_number), reason in page_exclusions.items():
    page_mask = (
        (page_records_df["document_id"] == document_id)
        & (page_records_df["page_number"] == page_number)
    )

    assert page_mask.sum() == 1

    page_records_df.loc[
        page_mask,
        "retrieval_eligible",
    ] = False

    page_records_df.loc[
        page_mask,
        "exclusion_reason",
    ] = reason

assert (~page_records_df["retrieval_eligible"]).sum() == 15

page_records_df.to_json(
    pages_output_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(
    page_records_df.groupby(
        ["retrieval_eligible", "exclusion_reason"],
        dropna=False,
    )
    .size()
)

In [ ]:
saved_pages_df = pd.read_json(
    pages_output_path,
    lines=True,
)

required_columns = {
    "document_id",
    "title",
    "document_type",
    "authority_level",
    "source_url",
    "page_number",
    "text",
    "retrieval_eligible",
    "exclusion_reason",
}

assert required_columns.issubset(saved_pages_df.columns)
assert len(saved_pages_df) == 201
assert saved_pages_df["retrieval_eligible"].sum() == 186
assert not saved_pages_df["text"].str.strip().eq("").any()
assert not saved_pages_df.duplicated(
    ["document_id", "page_number"]
).any()

gazette_pages = saved_pages_df[
    saved_pages_df["authority_level"] == "gazette"
]

report_pages = saved_pages_df[
    saved_pages_df["document_type"] == "consumer_report"
]

assert gazette_pages["effective_from"].notna().all()
assert report_pages["reporting_period_start"].notna().all()
assert report_pages["reporting_period_end"].notna().all()

print("Saved page dataset passed validation.")
print(f"Total pages: {len(saved_pages_df)}")
print(
    "Retrieval-eligible pages: "
    f"{saved_pages_df['retrieval_eligible'].sum()}"
)
print(
    "Excluded pages: "
    f"{(~saved_pages_df['retrieval_eligible']).sum()}"
)